# Attention Mechanisms — Build: Implement Flash Attention and Benchmark
> Section 01 — Topic 03 — build

**Prerequisites:** All attention mechanisms notebooks

**What you'll build:**
- Naive O(N²) attention implementation
- Tiled/Flash-style attention with O(N) memory
- Comprehensive benchmarks comparing both against PyTorch native

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time
import tracemalloc

torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")

## 1. Naive Attention Implementation

Our baseline implementation materializes the full $N \times N$ attention matrix. This is the straightforward approach — simple to understand but memory-hungry. For a sequence of length $N$ with hidden dimension $d$, this requires $O(N^2)$ memory for the attention weights matrix.

In [ ]:
def naive_attention(Q, K, V):
    """
    Standard attention that materializes the full N×N attention matrix.
    Memory: O(N^2) for the scores/weights matrices.
    """
    d_k = Q.shape[-1]
    
    # This creates an N×N matrix in memory
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    
    return output

# Verify correctness
N, d = 128, 64
Q = torch.randn(1, N, d)
K = torch.randn(1, N, d)
V = torch.randn(1, N, d)

output_naive = naive_attention(Q, K, V)
print(f"Input shapes: Q={Q.shape}, K={K.shape}, V={V.shape}")
print(f"Output shape: {output_naive.shape}")
print(f"Attention matrix size: {N}×{N} = {N*N:,} elements")
print(f"Attention matrix memory (fp32): {N*N*4/1024:.1f} KB")

## 2. Tiled/Flash Attention Implementation

The Flash Attention algorithm processes Q, K, V in **tiles** (blocks). Instead of computing the full attention matrix, it processes blocks of queries against blocks of keys, maintaining running statistics for the softmax normalization. The key algorithmic insight is the **online softmax** trick: we can compute softmax incrementally as new blocks arrive without needing to see all the scores at once.

For each block of queries, we iterate through all key blocks, computing:
1. Local attention scores for the current Q-block × K-block
2. Update running max (for numerical stability)
3. Rescale previous accumulated results to account for the new max
4. Add contribution of current block

This never materializes the full $N \times N$ matrix — only $B_q \times B_k$ blocks exist at any time, giving $O(N)$ memory.

In [ ]:
def tiled_flash_attention(Q, K, V, block_size=32):
    """
    Flash Attention-style tiled computation.
    Never materializes the full N×N attention matrix.
    Memory: O(N * block_size) instead of O(N^2).
    """
    batch, seq_len, d_k = Q.shape
    
    # Initialize output accumulators
    O = torch.zeros_like(Q)                                    # Output
    L = torch.zeros(batch, seq_len, 1)                          # Log-sum-exp (normalizer)
    M = torch.full((batch, seq_len, 1), float('-inf'))          # Running max
    
    scale = 1.0 / math.sqrt(d_k)
    
    # Number of K/V blocks
    n_kv_blocks = math.ceil(seq_len / block_size)
    
    # Process Q in blocks
    n_q_blocks = math.ceil(seq_len / block_size)
    
    for q_block_idx in range(n_q_blocks):
        q_start = q_block_idx * block_size
        q_end = min(q_start + block_size, seq_len)
        
        Q_block = Q[:, q_start:q_end, :]  # (batch, block_q, d_k)
        
        # Accumulators for this Q block
        O_block = torch.zeros(batch, q_end - q_start, d_k)
        m_block = torch.full((batch, q_end - q_start, 1), float('-inf'))  # running max
        l_block = torch.zeros(batch, q_end - q_start, 1)                   # running sum
        
        # Iterate through K/V blocks
        for kv_block_idx in range(n_kv_blocks):
            kv_start = kv_block_idx * block_size
            kv_end = min(kv_start + block_size, seq_len)
            
            K_block = K[:, kv_start:kv_end, :]  # (batch, block_k, d_k)
            V_block = V[:, kv_start:kv_end, :]  # (batch, block_k, d_k)
            
            # Compute scores for this tile only: (batch, block_q, block_k)
            S_block = torch.matmul(Q_block, K_block.transpose(-2, -1)) * scale
            
            # Online softmax: update running max and rescale
            m_block_new = torch.maximum(m_block, S_block.max(dim=-1, keepdim=True).values)
            
            # Rescale old values
            exp_diff = torch.exp(m_block - m_block_new)
            l_block = l_block * exp_diff
            O_block = O_block * exp_diff
            
            # Add new block's contribution
            P_block = torch.exp(S_block - m_block_new)  # (batch, block_q, block_k)
            l_block = l_block + P_block.sum(dim=-1, keepdim=True)
            O_block = O_block + torch.matmul(P_block, V_block)
            
            m_block = m_block_new
        
        # Normalize
        O[:, q_start:q_end, :] = O_block / l_block
    
    return O

# Verify correctness: should match naive attention
N, d = 128, 64
Q = torch.randn(1, N, d)
K = torch.randn(1, N, d)
V = torch.randn(1, N, d)

out_naive = naive_attention(Q, K, V)
out_tiled = tiled_flash_attention(Q, K, V, block_size=32)

max_diff = (out_naive - out_tiled).abs().max().item()
print(f"Max difference between naive and tiled: {max_diff:.2e}")
print(f"Numerically {'equivalent' if max_diff < 1e-5 else 'different'} ✓" if max_diff < 1e-5 else "")

## 3. Benchmarking Suite

Now let's benchmark both implementations across increasing sequence lengths to see the memory and speed differences.

In [ ]:
def measure_memory_and_time(fn, Q, K, V, n_warmup=2, n_runs=5, **kwargs):
    """Measure peak memory and execution time."""
    # Warmup
    for _ in range(n_warmup):
        _ = fn(Q, K, V, **kwargs)
    
    # Time
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = fn(Q, K, V, **kwargs)
        times.append(time.perf_counter() - start)
    
    # Memory (approximate via tracemalloc)
    tracemalloc.start()
    _ = fn(Q, K, V, **kwargs)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    return np.median(times) * 1000, peak / 1024  # ms, KB

# Benchmark
seq_lengths = [64, 128, 256, 512, 1024, 2048]
d_k = 64

results = {
    'naive_time': [], 'naive_mem': [],
    'tiled_time': [], 'tiled_mem': [],
    'sdpa_time': [], 'sdpa_mem': [],
}

for N in seq_lengths:
    Q = torch.randn(1, N, d_k)
    K = torch.randn(1, N, d_k)
    V = torch.randn(1, N, d_k)
    
    t, m = measure_memory_and_time(naive_attention, Q, K, V)
    results['naive_time'].append(t)
    results['naive_mem'].append(m)
    
    t, m = measure_memory_and_time(tiled_flash_attention, Q, K, V, block_size=64)
    results['tiled_time'].append(t)
    results['tiled_mem'].append(m)
    
    # PyTorch SDPA
    Q_4d = Q.unsqueeze(1)
    K_4d = K.unsqueeze(1)
    V_4d = V.unsqueeze(1)
    sdpa_fn = lambda q, k, v: F.scaled_dot_product_attention(q, k, v)
    t, m = measure_memory_and_time(sdpa_fn, Q_4d, K_4d, V_4d)
    results['sdpa_time'].append(t)
    results['sdpa_mem'].append(m)
    
    print(f"N={N:>5}: naive={results['naive_time'][-1]:.1f}ms, tiled={results['tiled_time'][-1]:.1f}ms, SDPA={results['sdpa_time'][-1]:.1f}ms")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(seq_lengths, results['naive_time'], 'o-', label='Naive O(N²)', linewidth=2)
ax1.plot(seq_lengths, results['tiled_time'], 's-', label='Tiled (Flash-style)', linewidth=2)
ax1.plot(seq_lengths, results['sdpa_time'], '^-', label='PyTorch SDPA', linewidth=2)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Execution Time')
ax1.legend()
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(seq_lengths, results['naive_mem'], 'o-', label='Naive O(N²)', linewidth=2)
ax2.plot(seq_lengths, results['tiled_mem'], 's-', label='Tiled (Flash-style)', linewidth=2)
ax2.plot(seq_lengths, results['sdpa_mem'], '^-', label='PyTorch SDPA', linewidth=2)
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Peak Memory (KB)')
ax2.set_title('Peak Memory Usage')
ax2.legend()
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Comparison with PyTorch Native

PyTorch's `scaled_dot_product_attention` automatically selects the best backend: Flash Attention (on compatible GPUs), memory-efficient attention, or the math fallback. Let's compare all available backends.

In [ ]:
# Compare PyTorch SDPA backends
N, d = 512, 64
Q = torch.randn(1, 1, N, d)  # (batch, heads, seq, d)
K = torch.randn(1, 1, N, d)
V = torch.randn(1, 1, N, d)

# Math backend (always available)
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_mem_efficient=False, enable_math=True):
    out_math = F.scaled_dot_product_attention(Q, K, V)

# Default (auto-select best)
out_auto = F.scaled_dot_product_attention(Q, K, V)

print(f"Math backend output shape: {out_math.shape}")
print(f"Auto backend output shape: {out_auto.shape}")
print(f"Outputs match: {torch.allclose(out_math, out_auto, atol=1e-5)}")
print(f"\nNote: On GPU with Flash Attention support, the auto backend")
print(f"will use Flash Attention which is 2-4x faster than math backend.")

## 5. Integration Test — Full Transformer Block

Let's plug both attention implementations into a complete transformer block and verify they produce identical results end-to-end.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, attn_fn):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.attn_fn = attn_fn  # Pluggable attention function
        
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def forward(self, x):
        # Self-attention with pluggable implementation
        h = self.norm1(x)
        qkv = self.W_qkv(h)
        Q, K, V = qkv.chunk(3, dim=-1)
        
        attn_out = self.attn_fn(Q, K, V)
        x = x + self.W_o(attn_out)
        
        # FFN
        x = x + self.ffn(self.norm2(x))
        return x

# Same weights, different attention implementations
d_model, n_heads = 128, 4
block_naive = TransformerBlock(d_model, n_heads, naive_attention)
block_tiled = TransformerBlock(d_model, n_heads, lambda q, k, v: tiled_flash_attention(q, k, v, block_size=32))

# Copy weights so they're identical
block_tiled.load_state_dict(block_naive.state_dict())

# Forward pass
x = torch.randn(2, 64, d_model)
with torch.no_grad():
    out_naive = block_naive(x)
    out_tiled = block_tiled(x)

max_diff = (out_naive - out_tiled).abs().max().item()
print(f"Full transformer block comparison:")
print(f"  Max difference: {max_diff:.2e}")
print(f"  Numerically equivalent: {max_diff < 1e-4}")
print(f"\nBoth implementations produce identical results when integrated")
print(f"into a complete transformer block. The tiled version just uses less memory.")

## Summary

**What we built:**
- Naive attention: simple but O(N²) memory — fine for short sequences
- Tiled Flash Attention: O(N) memory via online softmax and block processing
- Both produce numerically identical results

**Key insight:** Flash Attention doesn't change the math — it changes the **order of operations** to be IO-efficient. The actual GPU implementation goes further with kernel fusion and SRAM utilization, achieving 2-4x speedup on hardware.

**In practice:** Always use `torch.nn.functional.scaled_dot_product_attention` — it automatically selects the best backend for your hardware.

**Next:** [04-transformer-architectures/foundations.ipynb](../04-transformer-architectures/foundations.ipynb)